# Does classical image processing make ASL recognition robust to bad capture conditions?

**Problem.** Our ASL translator works well in a bright, clean room. Real users sign in dim bedrooms, in front of
bright windows, with cheap noisy webcams, over compressed video calls, and they move their hands. In those conditions the
hand landmark detector stops finding the hand, or finds it with displaced landmarks, and the letter classifier fails.

**Research question.** Under which degradations does recognition break, and which classical DIP pre-processing
(point transforms, histogram methods, spatial filters) restores it, at what computational cost?

**Method.**
1. Record real hand images for 24 static letters (`record_dataset.py`).
2. Apply controlled, repeatable degradations: low light, low light + sensor noise, Gaussian noise, motion blur, backlight, heavy JPEG.
3. Enhance each degraded frame with each candidate method, including an **adaptive pipeline** that measures the frame first
   (brightness, noise, contrast, sharpness) and only applies what it needs.
4. Measure **hand detection rate**, **end-to-end letter accuracy**, **landmark displacement** against the clean frame, **PSNR/SSIM**, and **runtime**.

**Pipeline.**
`camera frame → [quality measurement] → [denoise] → [exposure: gamma] → [local contrast: CLAHE] → [sharpen] → MediaPipe hand landmarks → 93-D geometric features → KNN → letter`

## 0. Configuration
Run `python record_dataset.py --session <name>` at least twice (different person and/or room) before running this notebook.
With a single session the notebook falls back to a temporal split, which is weaker evidence.

In [ ]:
DATASET_DIR = "dataset"          # created by record_dataset.py
TEST_SESSIONS = None             # e.g. ["bob_evening"]; None = last session is the test set
REAL_LOWLIGHT_SESSIONS = []      # optional: sessions genuinely recorded in a dim room (no synthetic degradation)
WORK_WIDTH = 480                 # frames are resized to this width (webcam-like, keeps NLM affordable)
MAX_TEST_PER_CLASS = 6           # test images per letter in the main experiment
SWEEP_PER_CLASS = 3              # images per letter for severity and parameter sweeps
CONF_THRESHOLD = 0.60            # same acceptance threshold as app_asl.py
RERUN = False                    # True = ignore cached results in results/ and recompute
FIG_DIR, RESULTS_DIR = "figures", "results"

In [ ]:
import os, time, glob, json
from functools import partial
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

import dip_preprocess as dp
from asl_core import HAND_CONNECTIONS, make_detector, detect_landmarks, extract_features, landmark_error

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 9, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8})
pd.set_option("display.precision", 3)

DEGS = list(dp.DEGRADATIONS)
NOISY = {"gaussian_noise", "low_light_noisy"}

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches="tight")

def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def load(path):
    img = cv2.imread(path)
    h, w = img.shape[:2]
    return cv2.resize(img, (WORK_WIDTH, int(round(h * WORK_WIDTH / w))), interpolation=cv2.INTER_AREA)

def apply_degradation(img, name, idx=0, **kw):
    if name in NOISY:
        kw.setdefault("seed", idx)   # different noise per image, reproducible
    return dp.degrade(img, name, **kw)

def draw_landmarks(img, lm):
    out = img.copy()
    if lm is None:
        return out
    h, w = out.shape[:2]
    pts = [(int(x * w), int(y * h)) for x, y, _ in lm]
    for s, e in HAND_CONNECTIONS:
        cv2.line(out, pts[s], pts[e], (0, 255, 0), 2)
    for p in pts:
        cv2.circle(out, p, 3, (0, 0, 255), -1)
    return out

def heatmap(table, title, fname, fmt="{:.0%}", cmap="viridis", vmin=0.0, vmax=1.0):
    fig, ax = plt.subplots(figsize=(1.2 + 0.8 * table.shape[1], 0.8 + 0.42 * table.shape[0]))
    im = ax.imshow(table.values.astype(float), cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=40, ha="right")
    ax.set_yticks(range(table.shape[0]), table.index)
    mid = (vmin + vmax) / 2
    for r in range(table.shape[0]):
        for c in range(table.shape[1]):
            v = table.values[r, c]
            if np.isfinite(v):
                dark_cell = v < mid if cmap == "viridis" else abs(v - mid) > (vmax - vmin) / 4
                ax.text(c, r, fmt.format(v), ha="center", va="center", fontsize=7,
                        color="white" if dark_cell else "black")
    fig.colorbar(im, ax=ax, fraction=0.03)
    ax.set_title(title)
    save(fig, fname)
    plt.show()

detector = make_detector(min_conf=0.5)

## 1. Dataset
Images come from our own webcam recordings (`dataset/<session>/<letter>/*.jpg`). The recorder keeps only frames in
which MediaPipe found a hand, and only every 4th such frame, to reduce near-duplicates.

**Why a session split?** Consecutive video frames are almost identical. A random frame-level split (what `train_knn.py`
does) puts near-copies of test frames into the training set and inflates accuracy. Testing on a different session
(another person or room) measures generalization.

In [ ]:
rows = []
for path in sorted(glob.glob(os.path.join(DATASET_DIR, "*", "*", "*.jpg"))):
    session, letter = path.split(os.sep)[-3:-1]
    rows.append({"session": session, "letter": letter, "path": path})
df = pd.DataFrame(rows)
assert len(df), f"No images found under {DATASET_DIR}/ - run record_dataset.py first."

sessions = sorted(df.session.unique())
real_ll = [s for s in REAL_LOWLIGHT_SESSIONS if s in sessions]
normal = [s for s in sessions if s not in real_ll]
test_s = list(TEST_SESSIONS) if TEST_SESSIONS else (normal[-1:] if len(normal) >= 2 else [])

if test_s:
    df["split"] = np.where(df.session.isin(test_s), "test", "train")
    SPLIT_NOTE = f"Session split: train on {[s for s in normal if s not in test_s]}, test on {test_s}"
else:
    order = df.groupby(["session", "letter"]).cumcount()
    n = df.groupby(["session", "letter"]).path.transform("count")
    df["split"] = np.where(order < 0.7 * n, "train", "test")
    SPLIT_NOTE = "Single session -> temporal split (first 70% of each letter train, last 30% test). Weaker evidence."
df.loc[df.session.isin(real_ll), "split"] = "real_lowlight"
LETTERS = sorted(df.letter.unique())

print(SPLIT_NOTE)
print(f"{len(df)} images, {len(LETTERS)} letters, sessions: {sessions}")
df.pivot_table(index="session", columns="split", values="path", aggfunc="count", fill_value=0)

In [ ]:
train_df = df[df.split == "train"]
cols = 8
rws = int(np.ceil(len(LETTERS) / cols))
fig, axes = plt.subplots(rws, cols, figsize=(cols * 1.6, rws * 1.4))
for ax in axes.flat:
    ax.axis("off")
for ax, letter in zip(axes.flat, LETTERS):
    sub = train_df[train_df.letter == letter]
    if len(sub):
        ax.imshow(rgb(load(sub.path.iloc[len(sub) // 2])))
    ax.set_title(letter, fontsize=10)
fig.suptitle("One training sample per letter", y=1.01)
save(fig, "01_dataset_samples.png")
plt.show()

## 2. Image-quality analysis of the clean data
Before choosing any filter we measure the frames. These no-reference measures are also what the adaptive pipeline uses:

| measure | definition | detects |
|---|---|---|
| brightness | mean luminance in [0, 1] | under / over exposure |
| contrast | std of luminance (RMS contrast) | flat, washed-out frames |
| noise σ | Immerkaer (1996): σ ≈ √(π/2) · Σ\|I∗M\| / (6(W−2)(H−2)), with M = [[1,−2,1],[−2,4,−2],[1,−2,1]] | sensor noise |
| sharpness | variance of the Laplacian | motion / focus blur |
| highlights | fraction of pixels ≥ 240 | glare, blown-out windows |

The adaptive thresholds are **derived from these clean statistics** instead of being guessed: a frame is only
"corrected" when it falls clearly outside the range of normal frames.

In [ ]:
q_paths = train_df.path.sample(min(300, len(train_df)), random_state=0)
q = pd.DataFrame([dp.quality_report(load(p)) for p in q_paths])
display(q.describe(percentiles=[0.05, 0.5, 0.95]).T)

fig, axes = plt.subplots(1, 5, figsize=(15, 2.4))
for ax, col in zip(axes, q.columns):
    ax.hist(q[col], bins=30, color="#4C72B0")
    ax.set_title(col)
fig.suptitle("Quality measures on clean training frames", y=1.04)
save(fig, "02_clean_quality_hist.png")
plt.show()

In [ ]:
TH = {
    "noise":     max(4.0, 1.5 * q.noise_sigma.quantile(0.95)),
    "dark":      min(0.35, 0.8 * q.brightness.quantile(0.05)),
    "bright":    max(0.65, 1.2 * q.brightness.quantile(0.95)),
    "contrast":  0.8 * q.contrast.quantile(0.05),
    "glare":     max(0.12, 1.5 * q.highlights.quantile(0.95)),
    "sharpness": 0.5 * q.sharpness.quantile(0.05),
}
dp.METHODS["adaptive (ours)"] = partial(dp.adaptive_enhance, thresholds=TH)
dp.METHODS["adaptive-fast (ours)"] = partial(dp.adaptive_enhance, thresholds=TH, denoiser="bilateral")
METHODS = list(dp.METHODS)

triggered = np.mean([len(dp.adaptive_enhance(load(p), True, TH)[1]) > 1 for p in q_paths[:100]])
with open(os.path.join(RESULTS_DIR, "adaptive_thresholds.json"), "w") as f:
    json.dump({k: float(v) for k, v in TH.items()}, f, indent=2)   # picked up by app_asl.py
print("Calibrated thresholds:", {k: round(v, 3) for k, v in TH.items()})
print(f"Adaptive pipeline modifies {triggered:.0%} of clean frames (ideally close to 0%).")

## 3. Degradation models
Each degradation models a real capture problem and is applied with a fixed, reproducible strength:

| name | model | real-world cause |
|---|---|---|
| low_light | out = 0.6 · in^γ, γ = 3 | dim room, short exposure |
| low_light_noisy | low light (γ = 2.5) + N(0, 12²) | dim room on a cheap sensor (gain amplifies noise) |
| gaussian_noise | in + N(0, 35²) | high ISO, poor webcam |
| motion_blur | in ∗ 21-px line kernel | fast hand movement, long exposure |
| backlight | 0.51 · in + 0.75 · radial glare | bright window behind the signer |
| jpeg | JPEG quality 6 | low-bandwidth video call |

In [ ]:
test_all = df[df.split == "test"]
ex_letter = "B" if "B" in set(test_all.letter) else test_all.letter.iloc[0]
example = load(test_all[test_all.letter == ex_letter].path.iloc[0])

fig, axes = plt.subplots(2, len(DEGS), figsize=(2.3 * len(DEGS), 3.4), gridspec_kw={"height_ratios": [3, 1.3]})
for c, name in enumerate(DEGS):
    img = apply_degradation(example, name)
    r = dp.quality_report(img)
    axes[0, c].imshow(rgb(img))
    axes[0, c].set_title(f"{name}\nmean {r['brightness']:.2f}  σ {r['noise_sigma']:.1f}\nsharp {r['sharpness']:.0f}")
    axes[0, c].axis("off")
    axes[1, c].hist(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).ravel(), bins=64, range=(0, 255), color="#555")
    axes[1, c].set_yticks([])
fig.suptitle("Degradations (top) and luminance histograms (bottom)", y=1.02)
save(fig, "03_degradations.png")
plt.show()

## 4. Enhancement methods
| method | type | why it might help | expected weakness |
|---|---|---|---|
| gamma (auto) | point transform, γ = log(0.45)/log(mean) | brightens dark frames without clipping | amplifies noise in shadows |
| hist. equalize | global histogram | stretches a compressed histogram | over-amplifies background, ignores local contrast |
| CLAHE | local, clip-limited histogram | local contrast, e.g. a dark hand against a bright window | needs a clip limit and tile size |
| gaussian 5×5 | linear low-pass | removes Gaussian noise | blurs finger edges |
| median 5×5 | order statistic | removes impulse noise, keeps edges better | rounds thin structures |
| bilateral | edge-preserving non-linear | smooths noise but not across finger boundaries | slow at large d, cartoon effect |
| NLM | patch-based non-local means | strongest denoiser for Gaussian noise | ~10× slower |
| adaptive (ours) | measure → denoise → gamma → CLAHE → sharpen | applies only what this frame needs | thresholds must be calibrated |

The order inside the adaptive pipeline is deliberate: denoise **before** brightening, because a gamma < 1 multiplies the
noise in dark regions; CLAHE comes **after** gamma, so it works on a correctly exposed histogram.

In [ ]:
show_methods = [m for m in METHODS if m != "none"]
for deg in ["low_light_noisy", "backlight"]:
    bad = apply_degradation(example, deg)
    cols = 6
    rws = int(np.ceil((len(show_methods) + 1) / cols))
    fig, axes = plt.subplots(rws, cols, figsize=(cols * 2.3, rws * 2.1))
    for ax in axes.flat:
        ax.axis("off")
    axes.flat[0].imshow(rgb(bad))
    axes.flat[0].set_title(f"degraded: {deg}")
    for ax, m in zip(axes.flat[1:], show_methods):
        out = dp.METHODS[m](bad)
        found = detect_landmarks(detector, out) is not None
        ax.imshow(rgb(out))
        ax.set_title(f"{m}\nhand {'found' if found else 'NOT found'}", color="green" if found else "red")
    fig.suptitle(f"Every method on the same {deg} frame", y=1.01)
    save(fig, f"04_methods_{deg}.png")
    plt.show()

In [ ]:
bad = apply_degradation(example, "low_light_noisy")
_, steps = dp.adaptive_enhance(bad, return_steps=True, thresholds=TH)
fig, axes = plt.subplots(2, len(steps), figsize=(2.8 * len(steps), 3.6), gridspec_kw={"height_ratios": [3, 1.3]})
axes = np.array(axes).reshape(2, -1)
for c, (name, img) in enumerate(steps):
    r = dp.quality_report(img)
    axes[0, c].imshow(rgb(draw_landmarks(img, detect_landmarks(detector, img))))
    axes[0, c].set_title(f"{c}. {name}\nmean {r['brightness']:.2f}  σ {r['noise_sigma']:.1f}  contrast {r['contrast']:.2f}")
    axes[0, c].axis("off")
    axes[1, c].hist(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).ravel(), bins=64, range=(0, 255), color="#555")
    axes[1, c].set_yticks([])
fig.suptitle("Intermediate outputs of the adaptive pipeline (landmarks drawn when detected)", y=1.03)
save(fig, "05_adaptive_steps.png")
plt.show()

## 5. Classifier (unchanged from the app)
Landmarks from the **clean** training frames → the same 93-D features and KNN (k = 3, distance-weighted) as
`app_asl.py`. The classifier is never retrained on degraded data, so every change in accuracy below comes from the image
processing stage alone.

In [ ]:
cache = os.path.join(RESULTS_DIR, "train_features.npz")
if os.path.exists(cache) and not RERUN:
    d = np.load(cache, allow_pickle=True)
    X_train, y_train = d["X"], d["y"]
else:
    X_train, y_train = [], []
    for p, letter in zip(train_df.path, train_df.letter):
        lm = detect_landmarks(detector, load(p))
        if lm is not None:
            X_train.append(extract_features(lm))
            y_train.append(letter)
    X_train, y_train = np.array(X_train), np.array(y_train)
    np.savez(cache, X=X_train, y=y_train)
print(f"{len(X_train)} training feature vectors ({len(train_df) - len(X_train)} frames lost after resizing)")

def pick(frame, n):
    idx = []
    for _, g in frame.groupby("letter"):
        sel = np.unique(np.linspace(0, len(g) - 1, min(n, len(g))).round().astype(int))
        idx.extend(g.index[sel])
    return frame.loc[idx].reset_index(drop=True)

test_df = pick(test_all, MAX_TEST_PER_CLASS)
test_imgs, test_labels, test_refs = [], [], []
for p, letter in zip(test_df.path, test_df.letter):
    img = load(p)
    lm = detect_landmarks(detector, img)
    if lm is not None:      # the clean landmarks are the reference for landmark displacement
        test_imgs.append(img); test_labels.append(letter); test_refs.append(lm)
print(f"{len(test_imgs)} test images with a clean reference detection")

X_test = np.array([extract_features(lm) for lm in test_refs])
k_rows = []
for k in [1, 3, 5, 7, 9]:
    m = KNeighborsClassifier(n_neighbors=k, weights="distance").fit(X_train, y_train)
    k_rows.append({"k": k, "clean test accuracy": (m.predict(X_test) == np.array(test_labels)).mean()})
display(pd.DataFrame(k_rows).set_index("k").T)

knn = KNeighborsClassifier(n_neighbors=3, weights="distance").fit(X_train, y_train)

def classify(lm):
    probs = knn.predict_proba([extract_features(lm)])[0]
    i = int(np.argmax(probs))
    return knn.classes_[i], float(probs[i])

### Classifier comparison
KNN is simple and needs no training, but is it the right choice? Same features, same session split, three classifiers:

| classifier | idea | parameters |
|---|---|---|
| KNN | vote of the 3 nearest training samples | k = 3, distance-weighted (as in the app) |
| SVM | maximum-margin boundary with an RBF kernel | C = 10, γ = "scale", features standardized |
| Random Forest | vote of 200 decision trees on random feature subsets | 200 trees |

*Per-frame latency* is the time for one `predict_proba` call, which is what the app runs on every webcam frame.

In [ ]:
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

y_test_clean = np.array(test_labels)
candidates = {
    "KNN (k=3)": KNeighborsClassifier(n_neighbors=3, weights="distance"),
    "SVM (RBF)": make_pipeline(StandardScaler(), CalibratedClassifierCV(SVC(C=10, gamma="scale"), ensemble=False)),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
}
clf_rows = []
for name, clf in candidates.items():
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0
    pred = clf.predict(X_test)
    t0 = time.perf_counter()
    for x in X_test[:50]:
        clf.predict_proba([x])
    frame_ms = (time.perf_counter() - t0) * 1000 / min(50, len(X_test))
    clf_rows.append({"classifier": name, "accuracy": (pred == y_test_clean).mean(),
                     "macro F1": f1_score(y_test_clean, pred, average="macro"),
                     "train time (s)": fit_s, "per-frame latency (ms)": frame_ms})
clf_table = pd.DataFrame(clf_rows).set_index("classifier")
clf_table.to_csv(os.path.join(RESULTS_DIR, "classifier_comparison.csv"))
display(clf_table)

fig, axes = plt.subplots(1, 2, figsize=(9, 2.6))
axes[0].barh(clf_table.index, clf_table.accuracy, color="#4C72B0")
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("clean test accuracy (unseen session)")
axes[1].barh(clf_table.index, clf_table["per-frame latency (ms)"], color="#DD8452")
axes[1].set_xlabel("ms per frame")
for ax in axes:
    ax.invert_yaxis()
axes[1].set_yticks([])
fig.suptitle("Classifier comparison on the same landmark features")
fig.tight_layout()
save(fig, "05b_classifier_comparison.png")
plt.show()
print("The rest of the notebook keeps KNN so results match app_asl.py. If another classifier wins clearly,"
      " swap it into train_knn.py and justify the change with this table.")

## 6. Main experiment: degradation × method
For every test image, every degradation and every method we record:
* **detected**: MediaPipe found a hand
* **accuracy**: correct letter, with a missed hand counted as wrong (what the user experiences)
* **accepted**: correct *and* confidence ≥ 0.6, which is what the app would actually type
* **landmark error**: mean landmark displacement versus the clean frame, in hand-size units
* **PSNR / SSIM** against the clean frame, plus processing time

In [ ]:
def evaluate(images, labels, refs, degradation, methods, deg_kw=None, measure_quality=True):
    """methods: dict name -> function(img); deg_kw: degradation parameters; refs may be None."""
    out = []
    for i, (img, label) in enumerate(zip(images, labels)):
        ref = refs[i] if refs is not None else None
        bad = apply_degradation(img, degradation, i, **(deg_kw or {})) if degradation else img
        for name, fn in methods.items():
            t0 = time.perf_counter()
            enh = fn(bad)
            ms = (time.perf_counter() - t0) * 1000
            lm = detect_landmarks(detector, enh)
            row = {"degradation": degradation, "method": name, "idx": i, "label": label, "ms": ms,
                   "detected": lm is not None, "pred": None, "conf": 0.0, "correct": False,
                   "accepted": False, "lm_err": np.nan, "psnr": np.nan, "ssim": np.nan}
            if measure_quality and refs is not None:
                row["psnr"], row["ssim"] = dp.psnr(img, enh), dp.ssim(img, enh)
            if lm is not None:
                pred, conf = classify(lm)
                row.update(pred=pred, conf=conf, correct=pred == label,
                           accepted=bool(pred == label and conf >= CONF_THRESHOLD))
                if ref is not None:
                    row["lm_err"] = landmark_error(lm, ref)
            out.append(row)
    return out

def summarize(res, by):
    return res.groupby(by, sort=False).agg(
        detection=("detected", "mean"), accuracy=("correct", "mean"), accepted=("accepted", "mean"),
        lm_err=("lm_err", "median"), psnr=("psnr", "mean"), ssim=("ssim", "mean"), ms=("ms", "mean"))

main_csv = os.path.join(RESULTS_DIR, "main_results.csv")
if os.path.exists(main_csv) and not RERUN:
    res = pd.read_csv(main_csv)
else:
    rows = []
    for deg in DEGS:
        t0 = time.time()
        rows += evaluate(test_imgs, test_labels, test_refs, deg, dp.METHODS)
        print(f"{deg:16s} done in {time.time() - t0:5.1f}s")
    res = pd.DataFrame(rows)
    res.to_csv(main_csv, index=False)

summary = summarize(res, ["degradation", "method"])
summary.to_csv(os.path.join(RESULTS_DIR, "main_summary.csv"))
acc = summary.accuracy.unstack().loc[DEGS, METHODS]
det = summary.detection.unstack().loc[DEGS, METHODS]
lme = summary.lm_err.unstack().loc[DEGS, METHODS]
print(f"{len(res)} evaluations")

In [ ]:
heatmap(det, "Hand detection rate", "06_heatmap_detection.png")
heatmap(acc, "End-to-end letter accuracy (missed hand = wrong)", "07_heatmap_accuracy.png")
gain = acc.sub(acc["none"], axis=0) * 100
lim = max(5.0, float(np.nanmax(np.abs(gain.values))))
heatmap(gain, "Accuracy gain over no pre-processing (percentage points)", "08_heatmap_gain.png",
        fmt="{:+.0f}", cmap="RdBu", vmin=-lim, vmax=lim)
heatmap(lme, "Median landmark displacement vs clean frame (hand-size units, lower is better)",
        "09_heatmap_landmark_error.png", fmt="{:.3f}", cmap="viridis",
        vmin=0, vmax=float(np.nanmax(lme.values)) if np.isfinite(lme.values).any() else 1)

In [ ]:
fixed = [m for m in METHODS if m not in ("none", "adaptive (ours)", "adaptive-fast (ours)")]
best_fixed = acc[fixed].idxmax(axis=1)
table = pd.DataFrame({
    "no processing": acc["none"],
    "best fixed method": best_fixed,
    "best fixed acc.": acc[fixed].max(axis=1),
    "adaptive": acc["adaptive (ours)"],
    "adaptive-fast": acc["adaptive-fast (ours)"],
    "adaptive gain (pp)": (acc["adaptive (ours)"] - acc["none"]) * 100,
})
display(table)

show = ["none", "adaptive-fast (ours)", "adaptive (ours)"]
x = np.arange(len(DEGS))
fig, ax = plt.subplots(figsize=(10, 3.2))
for j, m in enumerate(show):
    ax.bar(x + (j - 1) * 0.27, acc.loc[DEGS, m], 0.27, label=m)
ax.set_xticks(x, DEGS)
ax.set_ylim(0, 1)
ax.set_ylabel("letter accuracy")
ax.legend(ncol=3, loc="upper right")
ax.set_title("Accuracy per condition: no processing vs adaptive pre-processing")
save(fig, "10_bar_adaptive_vs_none.png")
plt.show()

### Does a "better-looking" image mean better recognition?
PSNR and SSIM measure fidelity to the clean frame. Here we check whether they predict task performance. If they don't,
a method should be judged by the downstream task, not by image-quality scores alone.

In [ ]:
pts = summary.reset_index()
pts = pts[pts.degradation != "clean"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, metric in zip(axes, ["psnr", "ssim"]):
    for deg, g in pts.groupby("degradation", sort=False):
        ax.scatter(g[metric], g.accuracy, s=18, label=deg)
    r = pts[[metric, "accuracy"]].corr().iloc[0, 1]
    ax.set_xlabel(metric.upper())
    ax.set_ylabel("letter accuracy")
    ax.set_title(f"{metric.upper()} vs accuracy (Pearson r = {r:.2f})")
axes[1].legend(loc="lower right")
save(fig, "11_quality_vs_accuracy.png")
plt.show()

## 7. Severity sweeps: where does recognition break?
Each degradation is swept from mild to extreme. Each panel compares no processing, the best fixed method for that
degradation (from section 6), and both adaptive variants. The **breaking point** is where the "none" curve collapses;
a useful pre-processing stage moves it to the right.

In [ ]:
sweep_df = pick(pd.DataFrame({"letter": test_labels, "i": range(len(test_labels))}), SWEEP_PER_CLASS)
S_IDX = list(sweep_df.i)
s_imgs = [test_imgs[i] for i in S_IDX]
s_labels = [test_labels[i] for i in S_IDX]
s_refs = [test_refs[i] for i in S_IDX]
print(f"{len(s_imgs)} images per sweep point")

sev_csv = os.path.join(RESULTS_DIR, "severity_results.csv")
if os.path.exists(sev_csv) and not RERUN:
    sev = pd.read_csv(sev_csv)
else:
    rows = []
    for deg, (_, _, sweep) in dp.DEGRADATIONS.items():
        if sweep is None:
            continue
        pname, values = sweep
        ms_ = list(dict.fromkeys(["none", best_fixed[deg], "adaptive-fast (ours)", "adaptive (ours)"]))
        for v in values:
            r = evaluate(s_imgs, s_labels, s_refs, deg, {m: dp.METHODS[m] for m in ms_},
                         deg_kw={pname: v}, measure_quality=False)
            for row in r:
                row.update(param=pname, level=v)
            rows += r
        print(f"{deg:16s} swept over {pname} = {values}")
    sev = pd.DataFrame(rows)
    sev.to_csv(sev_csv, index=False)

for metric, fname in [("correct", "12_severity_accuracy.png"), ("detected", "13_severity_detection.png")]:
    sweep_degs = list(sev.degradation.unique())
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.4))
    for ax, deg in zip(axes.flat, sweep_degs):
        g = sev[sev.degradation == deg]
        for m, gm in g.groupby("method", sort=False):
            curve = gm.groupby("level", sort=False)[metric].mean()
            ax.plot(curve.index, curve.values, marker="o", ms=3, label=m)
        ax.set_title(deg)
        ax.set_xlabel(g.param.iloc[0] + (" (lower = worse)" if deg == "jpeg" else ""))
        ax.set_ylim(-0.02, 1.02)
        if deg == "jpeg":
            ax.invert_xaxis()
        ax.legend(fontsize=7)
    for ax in axes.flat[len(sweep_degs):]:
        ax.axis("off")
    fig.suptitle("Letter accuracy vs severity" if metric == "correct" else "Hand detection rate vs severity")
    fig.tight_layout()
    save(fig, fname)
    plt.show()

## 8. Parameter studies
The course asks us to justify every parameter. Each study varies one parameter on a degradation it is meant to fix
and reports the task metric (accuracy) alongside image-quality metrics where relevant.

In [ ]:
def param_sweep(degradation, deg_kw, make_fn, values, quality=True):
    rows = []
    for v in values:
        r = pd.DataFrame(evaluate(s_imgs, s_labels, s_refs, degradation, {str(v): make_fn(v)}, deg_kw, quality))
        rows.append({"value": v, "detection": r.detected.mean(), "accuracy": r.correct.mean(),
                     "psnr": r.psnr.mean(), "ssim": r.ssim.mean(), "ms": r.ms.mean()})
    return pd.DataFrame(rows)

par_csv = os.path.join(RESULTS_DIR, "parameter_studies.pkl")
if os.path.exists(par_csv) and not RERUN:
    studies = pd.read_pickle(par_csv)
else:
    studies = {}
    noise_kw = {"sigma": 35}
    studies["NLM h"] = param_sweep("gaussian_noise", noise_kw,
        lambda h: (lambda img: dp.denoise_nlm(img, h=h)) if h else (lambda img: img), [0, 3, 5, 8, 12, 16, 20, 25])
    studies["bilateral sigmaColor"] = param_sweep("gaussian_noise", noise_kw,
        lambda s: lambda img: dp.denoise_bilateral(img, d=9, sigma_color=s, sigma_space=9), [10, 25, 50, 75, 100, 150, 200])
    studies["gaussian kernel"] = param_sweep("gaussian_noise", noise_kw,
        lambda k: lambda img: dp.denoise_gaussian(img, ksize=k), [1, 3, 5, 7, 9, 11, 15])
    studies["median kernel"] = param_sweep("gaussian_noise", noise_kw,
        lambda k: lambda img: dp.denoise_median(img, ksize=k), [1, 3, 5, 7, 9, 11, 15])
    studies["gamma target"] = param_sweep("low_light", {"gamma": 3.0},
        lambda t: lambda img: dp.auto_gamma(img, target=t), [0.2, 0.3, 0.4, 0.45, 0.5, 0.6, 0.7])
    clahe_rows = []
    for tile in [4, 8, 16]:
        r = param_sweep("backlight", {"strength": 0.75},
            lambda c: lambda img: dp.clahe(img, clip_limit=c, tile=tile), [1, 2, 3, 4, 6, 8], quality=False)
        r["tile"] = tile
        clahe_rows.append(r)
    studies["CLAHE"] = pd.concat(clahe_rows, ignore_index=True)
    pd.to_pickle(studies, par_csv)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for name in ["NLM h", "bilateral sigmaColor"]:
    s = studies[name]
    ax = axes[0] if name == "NLM h" else axes[1]
    ax.plot(s.value, s.accuracy, "o-", label="accuracy")
    ax.plot(s.value, s.detection, "s--", label="detection")
    ax.plot(s.value, s.ssim, "^:", label="SSIM")
    ax.set_xlabel(name)
    ax.set_ylim(0, 1.02)
    ax2 = ax.twinx()
    ax2.plot(s.value, s.psnr, color="gray", alpha=0.6)
    ax2.set_ylabel("PSNR (dB, grey)")
    ax.set_title(f"{name} on Gaussian noise σ=35  (best acc. at {s.value[s.accuracy.idxmax()]})")
    ax.legend(loc="lower right")
for name, style in [("gaussian kernel", "o-"), ("median kernel", "s-")]:
    s = studies[name]
    axes[2].plot(s.value, s.accuracy, style, label=f"{name.split()[0]} accuracy")
axes[2].set_xlabel("kernel size")
axes[2].set_ylim(0, 1.02)
axes[2].set_title("Linear (Gaussian) vs non-linear (median) smoothing")
axes[2].legend()
fig.tight_layout()
save(fig, "14_param_denoisers.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
s = studies["gamma target"]
axes[0].plot(s.value, s.accuracy, "o-", label="accuracy")
axes[0].plot(s.value, s.detection, "s--", label="detection")
axes[0].set_xlabel("auto-gamma target mean")
axes[0].set_ylim(0, 1.02)
axes[0].set_title("Auto-gamma target on low light (γ=3)")
axes[0].legend()
c = studies["CLAHE"].pivot(index="tile", columns="value", values="accuracy")
im = axes[1].imshow(c.values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
axes[1].set_xticks(range(c.shape[1]), c.columns)
axes[1].set_yticks(range(c.shape[0]), [f"{t}x{t}" for t in c.index])
for r_ in range(c.shape[0]):
    for c_ in range(c.shape[1]):
        axes[1].text(c_, r_, f"{c.values[r_, c_]:.0%}", ha="center", va="center", fontsize=7,
                     color="white" if c.values[r_, c_] < 0.5 else "black")
axes[1].set_xlabel("clip limit")
axes[1].set_ylabel("tile grid")
axes[1].set_title("CLAHE parameters on backlight: accuracy")
fig.colorbar(im, ax=axes[1], fraction=0.04)
fig.tight_layout()
save(fig, "15_param_gamma_clahe.png")
plt.show()

for name, s in studies.items():
    if name != "CLAHE":
        print(f"{name:22s} best value = {s.value[s.accuracy.idxmax()]}  (accuracy {s.accuracy.max():.0%})")
c_df = studies["CLAHE"].reset_index(drop=True)
best_c = c_df.loc[c_df.accuracy.idxmax()]
print(f"{'CLAHE':22s} best clip = {best_c['value']}, tile = {int(best_c['tile'])}  (accuracy {best_c['accuracy']:.0%})")

## 9. Which letters break? Confusion matrices
The extra column **∅** means no hand was detected. Compare clean frames, dark noisy frames without processing, and the
same frames after the adaptive pipeline.

In [ ]:
panels = [("clean", "none"), ("low_light_noisy", "none"), ("low_light_noisy", "adaptive (ours)")]
labels_cm = LETTERS + ["∅"]
fig, axes = plt.subplots(1, 3, figsize=(17, 5.6))
for ax, (deg, m) in zip(axes, panels):
    g = res[(res.degradation == deg) & (res.method == m)]
    pred = g.pred.fillna("∅").where(g.detected, "∅")
    cm = confusion_matrix(g.label, pred, labels=labels_cm, normalize="true")[: len(LETTERS)]
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels_cm)), labels_cm, fontsize=6)
    ax.set_yticks(range(len(LETTERS)), LETTERS, fontsize=6)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(f"{deg} / {m}\naccuracy {g.correct.mean():.0%}")
fig.tight_layout()
save(fig, "16_confusion_matrices.png")
plt.show()

clean_none = res[(res.degradation == "clean") & (res.method == "none") & res.detected & ~res.correct]
if len(clean_none):
    print("Most frequent confusions on clean frames:")
    display(clean_none.groupby(["label", "pred"]).size().sort_values(ascending=False).head(8).rename("count"))

## 10. Success and failure cases
**Successes:** frames where no processing failed but the adaptive pipeline recovered the correct letter.
**Failures:** frames the adaptive pipeline still gets wrong. These go into the limitations section of the report.

In [ ]:
def case_gallery(rows, title, fname, n=6):
    rows = rows.head(n)
    if not len(rows):
        print("no cases for:", title)
        return
    fig, axes = plt.subplots(2, len(rows), figsize=(2.6 * len(rows), 4.6), squeeze=False)
    for c, (_, r) in enumerate(rows.iterrows()):
        bad = apply_degradation(test_imgs[r.idx], r.degradation, r.idx)
        enh = dp.METHODS["adaptive (ours)"](bad)
        axes[0, c].imshow(rgb(draw_landmarks(bad, detect_landmarks(detector, bad))))
        axes[0, c].set_title(f"{r.degradation}\ntrue {r.label}", fontsize=8)
        lm = detect_landmarks(detector, enh)
        axes[1, c].imshow(rgb(draw_landmarks(enh, lm)))
        if lm is None:
            res_txt = "no hand"
        else:
            p, cf = classify(lm)
            res_txt = f"pred {p} ({cf:.0%})"
        axes[1, c].set_title(f"adaptive: {res_txt}", fontsize=8)
        for ax in axes[:, c]:
            ax.axis("off")
    fig.suptitle(title + "  (top: degraded input, bottom: after adaptive pipeline)", y=1.02)
    save(fig, fname)
    plt.show()

key = ["degradation", "idx"]
none_r = res[res.method == "none"].set_index(key)
ad_r = res[res.method == "adaptive (ours)"].set_index(key)
both = none_r[["correct"]].join(ad_r[["correct", "label"]], lsuffix="_none", rsuffix="_ad").reset_index()
both = both[both.degradation != "clean"]
wins = both[~both.correct_none & both.correct_ad].groupby("degradation", group_keys=False).head(1)
fails = both[~both.correct_ad].groupby("degradation", group_keys=False).head(1)
case_gallery(wins, "Recovered by pre-processing", "17_success_cases.png")
case_gallery(fails, "Still failing after pre-processing", "18_failure_cases.png")
print("Failures of the adaptive pipeline per degradation:")
display(both.groupby("degradation").correct_ad.apply(lambda s: (~s).sum()).rename("failed frames"))

## 11. Runtime: can it run live?
A 30 FPS webcam leaves about 33 ms per frame for pre-processing plus landmark detection.

In [ ]:
t_det = []
for img in test_imgs[:40]:
    t0 = time.perf_counter()
    detect_landmarks(detector, img)
    t_det.append((time.perf_counter() - t0) * 1000)
det_ms = float(np.mean(t_det))
ms = res[res.degradation != "clean"].groupby("method").ms.mean().loc[METHODS]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(METHODS, ms.values, label="pre-processing")
ax.barh(METHODS, det_ms, left=ms.values, color="#bbb", label=f"MediaPipe detection ({det_ms:.0f} ms)")
ax.axvline(33.3, color="red", ls="--", lw=1, label="30 FPS budget")
ax.axvline(66.7, color="orange", ls="--", lw=1, label="15 FPS budget")
ax.invert_yaxis()
ax.set_xlabel(f"ms per {WORK_WIDTH}-px frame")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4)
ax.set_title("Cost per frame")
save(fig, "19_runtime.png")
plt.show()
display(pd.DataFrame({"pre-processing ms": ms, "total ms": ms + det_ms, "max FPS": 1000 / (ms + det_ms)}))

## 12. Validation on genuinely dark recordings (optional)
Synthetic degradations are controllable, but are they realistic? If you recorded a session in a really dim room and
listed it in `REAL_LOWLIGHT_SESSIONS`, it is evaluated here with no synthetic degradation. If the method ranking matches
section 6, the synthetic model is credible.

In [ ]:
if not real_ll:
    print("No REAL_LOWLIGHT_SESSIONS configured: skipping. "
          "Record one with: python record_dataset.py --session <name>_dark  (lights off, screen light only)")
else:
    rl_df = pick(df[df.split == "real_lowlight"], MAX_TEST_PER_CLASS)
    rl_imgs = [load(p) for p in rl_df.path]
    rl = pd.DataFrame(evaluate(rl_imgs, list(rl_df.letter), None, None, dp.METHODS, measure_quality=False))
    rl_sum = summarize(rl, ["method"])[["detection", "accuracy", "accepted", "ms"]]
    rl_sum.to_csv(os.path.join(RESULTS_DIR, "real_lowlight_summary.csv"))
    display(rl_sum)
    print("Quality of the real dark frames:")
    display(pd.DataFrame([dp.quality_report(i) for i in rl_imgs]).describe().T[["mean", "min", "max"]])

## 13. Summary numbers for the report

In [ ]:
lines = [SPLIT_NOTE,
         f"Test images: {len(test_imgs)} ({len(LETTERS)} letters); evaluations in main experiment: {len(res)}",
         f"Clean accuracy (no processing): {acc.loc['clean', 'none']:.0%}; with adaptive: {acc.loc['clean', 'adaptive (ours)']:.0%}"]
for deg in DEGS[1:]:
    lines.append(f"{deg:16s}: none {acc.loc[deg, 'none']:.0%} -> adaptive {acc.loc[deg, 'adaptive (ours)']:.0%}"
                 f" | fast {acc.loc[deg, 'adaptive-fast (ours)']:.0%} | best fixed: {best_fixed[deg]} {acc.loc[deg, best_fixed[deg]]:.0%}")
avg_none = acc.loc[DEGS[1:], "none"].mean()
avg_ad = acc.loc[DEGS[1:], "adaptive (ours)"].mean()
lines.append(f"Mean accuracy over degraded conditions: {avg_none:.0%} -> {avg_ad:.0%} ({(avg_ad - avg_none) * 100:+.0f} pp)")
lines.append(f"Runtime: adaptive {ms['adaptive (ours)']:.0f} ms, adaptive-fast {ms['adaptive-fast (ours)']:.0f} ms, detection {det_ms:.0f} ms")
text = "\n".join(lines)
print(text)
with open(os.path.join(RESULTS_DIR, "summary.txt"), "w", encoding="utf-8") as f:
    f.write(text)
print("\nFigures:", sorted(os.listdir(FIG_DIR)))

## 14. Discussion (fill in from your numbers)
* **Which degradations hurt most?** Rank them by the "none" column of the accuracy heatmap and give the breaking points from section 7.
* **Which methods help, and why?** Link each win to the mechanism: gamma fixes exposure, CLAHE fixes local contrast under backlight, NLM/bilateral remove noise without destroying finger edges.
* **Which methods hurt?** Global histogram equalization and heavy smoothing often reduce accuracy on clean frames. Explain why: amplified background, blurred finger boundaries.
* **Image quality ≠ task quality.** Use section 6's scatter plot: does the PSNR-optimal NLM `h` match the accuracy-optimal one?
* **Cost.** Is the adaptive pipeline real-time? The fast variant trades how much accuracy for how much speed?
* **Limitations.** Synthetic degradations; small number of signers; motion blur cannot really be undone by sharpening (deconvolution would need the blur kernel); thresholds are calibrated on our own camera.
* **Future work.** Wiener / Richardson-Lucy deconvolution for motion blur, temporal denoising across video frames, training the classifier on augmented data, two-hand and dynamic letters (J, Z).